In [ ]:
import os
import pandas as pd
import ast
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import math
from scipy.stats import gaussian_kde

In [ ]:
# Function to safely evaluate the dictionary strings and handle NaNs
def safe_eval(val):
    try:
        return ast.literal_eval(val) if isinstance(val, str) else np.nan
    except (SyntaxError, ValueError):
        return np.nan


# Extract Completeness and Conciseness scores
def extract_score(data, key):
    return data.get(key, np.nan) if isinstance(data, dict) else np.nan


In [ ]:
import ast

# Function to extract and compute required values
def extract_metrics(column, prompt='comple'):
    completeness, conciseness, avg, ratio = [], [], [], []
    count_both_1 = 0
    valid_points = 0
    total = 0

    for item in column.dropna():
        total += 1
        try:
            score_dict = ast.literal_eval(item)

            # Extract scores based on prompt type
            if prompt == 'comple':
                c, s = score_dict['Completeness'], score_dict['Conciseness']
                s = min(s, 1)  # Cap conciseness at 1
            elif prompt == 'concise':
                s, c = score_dict['Completeness'], score_dict['Conciseness']
                c = min(c, 1)  # Cap conciseness at 1
            elif prompt == 'balance':
                c, s = score_dict['Completeness'], score_dict['Conciseness']
                s = min(s, 1)  # Cap conciseness at 1
            else:
                continue  # Skip unknown prompt types

            # In 'comple' and 'concise' mode, skip perfect (1,1) pairs
            if prompt != 'balance' and c == 1 and s == 1:
                count_both_1 += 1
                continue

            # Valid points: both non-zero
            if c != 0 and s != 0:
                valid_points += 1
                if prompt == 'concise':
                    completeness.append(s)
                    conciseness.append(c)
                    
                else:
                    completeness.append(c)
                    conciseness.append(s)

                ratio.append(math.log(c / s))
                # avg.append((c + s) / 2)
                avg.append((2/(1/c + 1/s)))

        except Exception:
            continue

    # Compute metrics
    if prompt == 'balance':
        proportion_of_1 = -1  # Not applicable
    else:
        proportion_of_1 = count_both_1 / total if total else 0

    succ_rate = valid_points / total if total else 0

    return completeness, conciseness, avg, ratio, succ_rate, proportion_of_1


In [ ]:
# Plotting function
def stat_table(df, title):
    if 'MoreComple' in title:
        completeness, conciseness, avg, ratio, succ_rate, proportion_of_1 = extract_metrics(df,prompt='comple')
    elif 'MoreConcise' in title:
        completeness, conciseness, avg, ratio, succ_rate, proportion_of_1 = extract_metrics(df,prompt='concise')
    elif 'Balance' in title:
        completeness, conciseness, avg, ratio, succ_rate, proportion_of_1 = extract_metrics(df,prompt='balance')

    data_to_plot = [completeness, conciseness, avg, ratio, proportion_of_1]
    means = [round(np.mean(lst), 2) for lst in data_to_plot]

    # labels = ['Completeness', 'Conciseness', 'Average', 'Ratio', 'Proportion_of_1']
    
    # Print mean and median
    # for i, label in enumerate(labels):
    #     if data_to_plot[i]:  # Avoid empty lists
    #         print(f'{title} - {label}: Mean = {np.mean(data_to_plot[i]):.4f}, Median = {np.median(data_to_plot[i]):.4f}')
    #     else:
    #         print(f'{title} - {label}: No valid data')
    # print(f'Succesful rate:{succ_rate:.4f}')

    return means, round(np.mean(succ_rate), 2)


In [ ]:
csv_file_path = 'test_Llama_bs.csv'
df_llama_bs = pd.read_csv(csv_file_path)

csv_file_path = 'test_Llama_sota.csv'
df_llama_sota = pd.read_csv(csv_file_path)

csv_file_path = 'test_Llama_our.csv'
df_llama_ours = pd.read_csv(csv_file_path)

csv_file_path = 'test_qwen7_bs.csv'
df_qwen_bs = pd.read_csv(csv_file_path)

csv_file_path = 'test_qwen7_our.csv'
df_qwen_ours = pd.read_csv(csv_file_path)

csv_file_path = 'test_Mistral-7B_bs.csv'
df_mistral_bs = pd.read_csv(csv_file_path)

csv_file_path = 'test_Mistral-7B_our.csv'
df_mistral_ours = pd.read_csv(csv_file_path)

csv_file_path = 'test_gpt4o.csv'
df_gpt4o = pd.read_csv(csv_file_path)


In [ ]:
# Dictionary of all your dataframes
df_dict = {
    'llama_bs': df_llama_bs,
    'llama_sota': df_llama_sota,
    'llama_ours': df_llama_ours,
    'qwen_bs': df_qwen_bs,
    'qwen_ours': df_qwen_ours,
    'mistral_bs': df_mistral_bs,
    'mistral_ours': df_mistral_ours,
    'gpt4o': df_gpt4o,
}

# The categories to analyze
categories = ['Epoch1_score_MoreComple', 'Epoch1_score_MoreConcise', 'Epoch1_score_Balance']

# Generate stats and plots
for name, df in df_dict.items():
    print(f'Name:{name}')
    print(f"{'Category':<12}|{'Complete':>9}|{'Concise':>9}|{'Average':>9}|{'Ratio':>7}|{'Succ rate':>10}|{'Prop. of 1':>13}")
    print('--------------------------------------------------------------------------')
    for category in categories:
        if category in df.columns:  # safety check
            (Complete, Concise, Average, Ratio, Proportion_of_1), succ_rate = stat_table(df[category], title=f"{name} - {category}")
            category_ = category.split('_')[-1]
            print(f"{category_:<12}|{Complete:9.2f}|{Concise:9.2f}|{Average:9.2f}|{Ratio:7.2f}|{succ_rate:10.2f}|{Proportion_of_1:13.2f}")
            
    print('###########################################################################')

# plot contour figure

In [ ]:
import math

# Function to extract and compute required values
def extract_metrics_harmonic(column, prompt='comple'):

    completeness, conciseness, avg, ratio = [], [], [], []
    count_both_1 = 0
    valid_points = 0
    total = 0

    for item in column.dropna():
        total += 1
        try:
            score_dict = ast.literal_eval(item)

            # Extract scores based on prompt type
            if prompt == 'comple':
                c, s = score_dict['Completeness'], score_dict['Conciseness']
                s = min(s, 1)  # Cap conciseness at 1
            elif prompt == 'concise':
                s, c = score_dict['Completeness'], score_dict['Conciseness']
                c = min(c, 1)  # Cap conciseness at 1
            elif prompt == 'balance':
                c, s = score_dict['Completeness'], score_dict['Conciseness']
                s = min(s, 1)  # Cap conciseness at 1
            else:
                continue  # Skip unknown prompt types

            # In 'comple' and 'concise' mode, skip perfect (1,1) pairs
            if prompt != 'balance' and c == 1 and s == 1:
                count_both_1 += 1
                continue

            # Valid points: both non-zero
            if c != 0 and s != 0:
                valid_points += 1
                if prompt == 'concise':
                    completeness.append(s)
                    conciseness.append(c)
                else:
                    completeness.append(c)
                    conciseness.append(s)

                ratio.append(math.log((c/ s)))
                avg.append(2/(1/c + 1/s))

        except Exception:
            continue

    # Compute metrics
    if prompt == 'balance':
        proportion_of_1 = -1  # Not applicable
    else:
        proportion_of_1 = count_both_1 / total if total else 0

    succ_rate = valid_points / total if total else 0

    return completeness, conciseness, avg, ratio, succ_rate, proportion_of_1


In [ ]:
def plot_contour_harm(x, y, title):
    """
    Generate and display a contour plot for the given x and y data.

    Parameters:
    x (array-like): Data for the x-axis (Ratio).
    y (array-like): Data for the y-axis (Average).
    title (str): Title of the plot.
    """
    # Estimate the density using KDE
    xy = np.vstack([x, y])
    z = gaussian_kde(xy)(xy)  # Density estimation

    # Create a 2D grid for contour plotting
    xi, yi = np.meshgrid(np.linspace(-2, 2, 100), np.linspace(0, 1, 100))  # Set fixed grid based on limits
    # xi, yi = np.meshgrid(np.linspace(x.min(), x.max(), 100),
    #                      np.linspace(y.min(), y.max(), 100))
    zi = gaussian_kde(xy)(np.vstack([xi.ravel(), yi.ravel()])).reshape(xi.shape)

    # Plot contour
    plt.figure(figsize=(8, 6))
    contour = plt.contourf(xi, yi, zi, levels=20, cmap='coolwarm')
    plt.colorbar(contour)

    # Labels and title
    plt.xlabel("Log Ratio")
    plt.ylabel("Harmonic Mean")
    plt.title(title)

    # Set x and y limits
    plt.xlim(-2, 2)
    plt.ylim(0, 1)

    plt.show()

# baseline

In [ ]:
# Generate plots for each category
for category in ['Epoch1_score_MoreComple', 'Epoch1_score_MoreConcise', 'Epoch1_score_Balance']:
    if category=='Epoch1_score_MoreConcise':  prompt='concise'
    else: prompt='comple'
    completeness, conciseness, avg, ratio, succ_rate, _ = extract_metrics_harmonic(df_llama_bs[category], prompt)
    plot_contour_harm(np.array(ratio), np.array(avg), f"Contour Plot of {category} (Log Ratio vs. Harmonic Average)")